# Projekt 03 (final) — Kundensegmentierung

**Modul 05 — Machine Learning 2 · Abschlussprojekt (Unsupervised Learning)**

Dies ist das **Abschlussprojekt** des Moduls. Anders als bei den ersten beiden Projekten gibt es hier **keinen vorgeschriebenen Code** — du wendest alles selbst an: Vorverarbeitung, Dimensionsreduktion, vier Clustering-Verfahren, Modellwahl und Interpretation. Nur die reine Dateninfrastruktur (Download) ist vorgegeben, damit du sofort loslegen kannst.

Arbeite die Abschnitte der Reihe nach ab. Zu jedem gehört eine **leere Code-Zelle** (dein Code) und danach eine **Reflexionsfrage**, die du schriftlich beantwortest. Vergleiche erst zum Schluss mit `loesung/loesung.ipynb`.

---

### Die Aufgabe

Ein portugiesischer Großhandels-Distributor kennt seine 440 Kunden nur über ihre **jährlichen Ausgaben in sechs Produktkategorien** (`Fresh`, `Milk`, `Grocery`, `Frozen`, `Detergents_Paper`, `Delicassen`). Deine Aufgabe:

> **Finde datengetrieben natürliche Kundensegmente. Beantworte: Wie viele gibt es, wie sehen sie aus, welches Verfahren beschreibt sie am besten — und wie überzeugst du dich (und den Distributor), dass die Segmente real sind?**

**Zwei Spalten sind tabu fürs Clustering:** `Channel` (Horeca vs. Retail) und `Region`. Die hebst du als **externe Validierung** auf: Ein gutes, rein aus Ausgaben gewonnenes Clustering sollte die `Channel`-Struktur wiederentdecken, *ohne sie je gesehen zu haben*. Genau das misst du am Ende mit dem Adjusted Rand Index.

**Warum dieses Format (Notebook):** Segmentierung lebt vom Wechselspiel aus Plot, Kennzahl und Interpretation direkt nebeneinander.
**Warum echte Daten (UCI Wholesale Customers):** kompakt (440×6), aber mit allen realen Tücken — starke Rechtsschiefe, überlappende Cluster, eine externe Wahrheit zum Validieren. Ideal, um die Verfahren des Moduls *gegeneinander* zu stellen.

## Setup (vorgegeben)

Lädt die Daten nach `daten/` (gecacht, per `.gitignore` ausgeschlossen) und trennt Merkmale von den externen Labels. **Ab hier schreibst du selbst.**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import urllib.request

RNG = 42
np.random.seed(RNG)

DATA_DIR = Path("daten"); DATA_DIR.mkdir(exist_ok=True)
CSV = DATA_DIR / "wholesale_customers.csv"
URL = ("https://archive.ics.uci.edu/ml/machine-learning-databases/"
       "00292/Wholesale%20customers%20data.csv")
if not CSV.exists():
    req = urllib.request.Request(URL, headers={"User-Agent": "Mozilla/5.0"})
    CSV.write_bytes(urllib.request.urlopen(req, timeout=60).read())

df = pd.read_csv(CSV)
FEATURES = ["Fresh", "Milk", "Grocery", "Frozen", "Detergents_Paper", "Delicassen"]

# Externe Labels sichern (NICHT fürs Clustering verwenden!)
channel = df["Channel"].values - 1     # 0 = Horeca, 1 = Retail
region  = df["Region"].values
X = df[FEATURES].values.astype(float)
print("X:", X.shape, "| Channel-Verteilung:", np.bincount(channel))
df.head()

## Aufgabe 1 — EDA & Vorverarbeitung

Sieh dir die Verteilungen der sechs Merkmale an. Begründe, **warum rohes Clustering scheitern würde**, und leite daraus deine Vorverarbeitung ab.

**Zu tun:**
- Histogramme (oder Boxplots) der sechs Merkmale; berechne die Schiefe (`scipy.stats.skew`).
- Transformiere sinnvoll (Stichwort: multiplikative, log-normale Größen) und standardisiere anschließend.
- Lege dir das vorverarbeitete Array (z. B. `Xs`) an — darauf läuft alles Weitere.

In [ ]:
# dein Code


> **Reflexion 1:** Warum ist eine Log-Transformation *vor* der Standardisierung hier wichtig? Was würde bei k-Means passieren, wenn du auf den Rohdaten clusterst? *(Antwort:)*

## Aufgabe 2 — Dimensionsreduktion & Visualisierung

**Zu tun:**
- **PCA** auf `Xs`. Scree-Plot (wie viel Varianz in PC1+PC2?). Erstelle einen **Biplot** und färbe die Punkte nach dem *zurückgehaltenen* `channel` ein.
- Interpretiere die **Ladungen** von PC1 und PC2: Welche Produktkategorien hängen zusammen, und was bedeutet das inhaltlich?
- Optional: **t-SNE** (`perplexity≈30`) zur Visualisierung. Wie sieht die Cluster-Trennung aus — scharfe Lücken oder fließender Übergang?

In [ ]:
# dein Code


> **Reflexion 2:** Was sagen dir die PC-Ladungen über die *Art* der Kunden? Warum darfst du t-SNE-Distanzen **nicht** wie euklidische Distanzen interpretieren? *(Antwort:)*

## Aufgabe 3 — Vier Clustering-Verfahren

Wende **alle vier** Verfahren des Moduls an und wähle für jedes seine Hyperparameter *sauber begründet* (nicht raten):

1. **k-Means** — bestimme $k$ über **Elbow** *und* **Silhouette**.
2. **Gaussian Mixture (EM)** — wähle Komponentenzahl **und** Kovarianztyp über den **BIC**. Sieh dir auch die weichen Zuordnungen (`predict_proba`) an.
3. **DBSCAN** — `min_samples` per Heuristik ($\approx 2\cdot\dim$), `eps` über den **k-Distanz-Plot**. Was passiert? Lass dich vom Ergebnis nicht überraschen, sondern *erkläre* es.
4. **Agglomerativ (Ward)** — zeichne das **Dendrogramm** und lies die natürliche Cluster-Zahl ab.

In [ ]:
# dein Code — k-Means (Elbow + Silhouette)


In [ ]:
# dein Code — GMM/EM (BIC über Komponenten & Kovarianztyp)


In [ ]:
# dein Code — DBSCAN (k-Distanz-Plot, eps-Sweep)


In [ ]:
# dein Code — Ward (Dendrogramm)


> **Reflexion 3:** Auf welche Cluster-Zahl deuten die drei "kompakten" Verfahren (k-Means, GMM, Ward) übereinstimmend? Und warum liefert DBSCAN ein völlig anderes Bild — welche seiner Grundannahmen ist bei diesen Daten verletzt? *(Antwort:)*

## Aufgabe 4 — Validierung: intern **und** extern

Baue eine **Vergleichstabelle** über deine besten Lösungen aller vier Verfahren mit:
- **internen** Maßen: Silhouette (↑), Davies-Bouldin (↓), Calinski-Harabasz (↑),
- dem **externen** Maß: **Adjusted Rand Index** gegen den zurückgehaltenen `channel`.

Achtung: Internes Optimum (z. B. bester BIC) und externes Optimum (bester ARI) müssen **nicht** dasselbe Verfahren / dieselbe Cluster-Zahl sein — das ist Teil der Erkenntnis.

In [ ]:
# dein Code — Vergleichstabelle (silhouette/DB/CH + ARI gegen channel)


> **Reflexion 4:** Welches Verfahren entdeckt die `Channel`-Struktur am treusten (höchster ARI) — und *warum* passt gerade dessen Modellannahme zu dieser Datenform? Warum reicht ein einzelnes Gütemaß nie aus? *(Antwort:)*

## Aufgabe 5 — Segmente benennen & Geschäftsempfehlung

Nimm deine **Gewinner-Lösung** und mache sie geschäftlich nutzbar:
- Beschreibe jedes Segment über seine **Median-Ausgaben pro Kategorie** (zurück im Geld-Raum, nicht im z-Score).
- Gib jedem Segment einen **sprechenden Namen** und eine kurze Charakterisierung.
- Zeige die PCA-Karte zweimal nebeneinander: links deine gefundenen Segmente, rechts der wahre `channel`.
- Formuliere **eine konkrete Handlungsempfehlung** für den Distributor (Logistik / Sortiment / Marketing).

In [ ]:
# dein Code — Segmentprofile + finale Visualisierung


> **Reflexion 5 (Abschluss):** Fasse in 4–5 Sätzen zusammen: Wie viele Segmente, welches Verfahren, wie validiert, welche Geschäftsempfehlung? Und die Meta-Erkenntnis: Wovon hängt beim unüberwachten Lernen ab, *welche* Antwort du bekommst? *(Antwort:)*

---
## Abnahmekriterien (woran du merkst, dass du fertig bist)

- [ ] Vorverarbeitung begründet (log + z-Score), rohes vs. transformiertes Bild gezeigt.
- [ ] PCA-Ladungen inhaltlich interpretiert; 2D-Visualisierung vorhanden.
- [ ] Alle **vier** Verfahren angewandt, Hyperparameter je **begründet** gewählt (Elbow/Silhouette, BIC, k-Distanz, Dendrogramm).
- [ ] DBSCANs Verhalten **erklärt** (nicht nur berichtet).
- [ ] Vergleichstabelle mit internen Maßen **und** ARI gegen `channel`.
- [ ] Segmente benannt, Median-Profile gezeigt, eine Geschäftsempfehlung formuliert.
- [ ] Alle fünf Reflexionsfragen schriftlich beantwortet.

**Referenzwerte** (zur groben Orientierung — leichte Abweichungen sind normal): k-Means/GMM/Ward konvergieren auf **~2** Hauptsegmente; der beste ARI gegen `Channel` liegt bei **~0,6** (erreicht vom GMM mit voller Kovarianz); DBSCAN findet **keine** sinnvolle Struktur.